# PEFT(LoRA) + Fine-tuning — Linear Evaluation 대비 성능 비교

> 9차 미팅 `🧠 [학습] PEFT + Fine-tuning` · Stage 4 (비교/보고)

각 데이터셋(`aptos2019`, `Oral_Diseases`)에 대해 4가지 방법을 **동일 test split·동일 지표 정의**로 비교한다:
1. **Linear Eval** (baseline, frozen CLS + linear probe)
2. **LoRA (순수)** — 마지막 CLS + linear head
3. **LoRA (융합)** — 중간층 [11,15,19,23] 결합
4. **Full FT** — `run_class_finetuning.py` (--weights --monitor recall --layer_decay 0.65)

지표: BACC / Macro AUPR / AUROC(ovo) / W-F1 / Acc + 소수 클래스 recall. test 셋이 작으므로(oral=54) bootstrap CI 로 유의성을 함께 본다.

> **주의**: LoRA/Full-FT 결과 파일은 집의 RTX 3070 에서 학습을 완료해야 채워진다. 아직 없는 항목은 자동으로 `TBD` 로 표시된다.

## 0. 로드

In [1]:
import importlib
from pathlib import Path
import numpy as np
import pandas as pd
import peft_ft_utils as U
importlib.reload(U)

OUTPUT_ROOT = U.OUTPUT_ROOT
DATASETS = ["aptos2019", "oral_diseases"]
print("output_dir:", OUTPUT_ROOT)

output_dir: /home/junkim/paper_ajou_dev/PanDerm/output_dir


## 1. 방법별 결과 수집

- baseline: `<ds>_panderm_large_lp_result.csv`
- LoRA(순수/융합): `<ds>_{lora_lp,lora_multilayer_lp}/comparison_vs_baseline.csv` + `*_test_predictions.csv`(소수 클래스 recall·CI 용)
- Full FT: `<ds>_full_ft/test.csv` → `metrics_from_prediction_csv` 로 재계산

In [2]:
def safe_metrics_from_pred(path, K):
    p = Path(path)
    if not p.exists():
        return None
    try:
        m, ci = U.metrics_from_prediction_csv(p, K)
        return {"metrics": m, "ci": ci}
    except Exception as e:
        print(f"[warn] {path}: {e}")
        return None

def collect(dsk):
    cfg = U.DATASETS[dsk]
    K = cfg["num_classes"]
    rows = []
    base = U.load_baseline_row(cfg)
    rows.append({"method": "Linear Eval (baseline)", **base, "src": "anchor"})
    # LoRA pure / fusion
    for variant, suffix, name in [("pure", "lora_lp", "LoRA (pure, last CLS)"),
                                   ("fusion", "lora_multilayer_lp", "LoRA (fusion [11,15,19,23])")]:
        pred = OUTPUT_ROOT / f"{dsk}_{suffix}" / f"{dsk}_{variant}_test_predictions.csv"
        r = safe_metrics_from_pred(pred, K)
        if r:
            m = r["metrics"]
            rows.append({"method": name, "bacc": m["bacc"], "auroc": m["auroc"],
                         "aupr": m["aupr"], "acc": m["acc"], "weighted_f1": m["weighted_f1"],
                         "src": "pred_csv"})
        else:
            rows.append({"method": name, "bacc": np.nan, "auroc": np.nan, "aupr": np.nan,
                         "acc": np.nan, "weighted_f1": np.nan, "src": "TBD"})
    # Full FT (run_class_finetuning test.csv)
    pred = OUTPUT_ROOT / f"{dsk}_full_ft" / "test.csv"
    r = safe_metrics_from_pred(pred, K)
    if r:
        m = r["metrics"]
        rows.append({"method": "Full FT", "bacc": m["bacc"], "auroc": m["auroc"],
                     "aupr": m["aupr"], "acc": m["acc"], "weighted_f1": m["weighted_f1"], "src": "pred_csv"})
    else:
        rows.append({"method": "Full FT", "bacc": np.nan, "auroc": np.nan, "aupr": np.nan,
                     "acc": np.nan, "weighted_f1": np.nan, "src": "TBD"})
    return pd.DataFrame(rows).set_index("method")

tables = {d: collect(d) for d in DATASETS}
for d, t in tables.items():
    print(f"\n===== {d} =====")
    print(t.round(4))

/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one


===== aptos2019 =====
                               bacc   auroc    aupr     acc  weighted_f1  \
method                                                                     
Linear Eval (baseline)       0.6280  0.8957  0.6791  0.8141       0.8060   
LoRA (pure, last CLS)        0.6917  0.9051  0.7144  0.7834       0.7893   
LoRA (fusion [11,15,19,23])  0.6736  0.8996  0.6891  0.7834       0.7894   
Full FT                      0.6921  0.9051  0.7040  0.7960       0.8004   

                                  src  
method                                 
Linear Eval (baseline)         anchor  
LoRA (pure, last CLS)        pred_csv  
LoRA (fusion [11,15,19,23])  pred_csv  
Full FT                      pred_csv  

===== oral_diseases =====
                               bacc   auroc    aupr     acc  weighted_f1  \
method                                                                     
Linear Eval (baseline)       0.8107  0.9463  0.7981  0.7778       0.7584   
LoRA (pure, last CLS)    

## 2. 소수 클래스 recall + bootstrap CI

aptos: severe(3)/proliferative(4), oral: OLP(5). baseline recall 은 문서/노트북 값과 대조.

In [3]:
def minority_recall(dsk):
    cfg = U.DATASETS[dsk]; K = cfg["num_classes"]
    out = {}
    for variant, suffix in [("pure", "lora_lp"), ("fusion", "lora_multilayer_lp")]:
        pred = OUTPUT_ROOT / f"{dsk}_{suffix}" / f"{dsk}_{variant}_test_predictions.csv"
        r = safe_metrics_from_pred(pred, K)
        if r:
            pcr = r["metrics"]["per_class_recall"]
            out[f"LoRA-{variant}"] = {name: pcr[str(idx)] for name, idx in cfg["minority_classes"].items()}
    ft = OUTPUT_ROOT / f"{dsk}_full_ft" / "test.csv"
    r = safe_metrics_from_pred(ft, K)
    if r:
        pcr = r["metrics"]["per_class_recall"]
        out["Full-FT"] = {name: pcr[str(idx)] for name, idx in cfg["minority_classes"].items()}
    return out

for d in DATASETS:
    print(f"[{d}] 소수 클래스 recall:", minority_recall(d))

[aptos2019] 소수 클래스 recall: {'LoRA-pure': {'severe': 0.5666666666666667, 'proliferative_dr': 0.5333333333333333}, 'LoRA-fusion': {'severe': 0.6, 'proliferative_dr': 0.37777777777777777}, 'Full-FT': {'severe': 0.6666666666666666, 'proliferative_dr': 0.4888888888888889}}


/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one

[oral_diseases] 소수 클래스 recall: {'LoRA-pure': {'OLP': 0.3}, 'LoRA-fusion': {'OLP': 0.3}, 'Full-FT': {'OLP': 0.3}}


## 3. 회의록 2-3절용 표 저장

`PanDerm/output_dir/peft_ft_comparison_<ds>.csv` 로 저장 → 회의록 표에 그대로 반영.

In [4]:
for d, t in tables.items():
    path = OUTPUT_ROOT / f"peft_ft_comparison_{d}.csv"
    t.round(4).to_csv(path)
    print("saved", path)

# 완료 후: docs/date/2026-07-10_9th_meeting/minutes/2026-07-10_9th_meeting.md 2-3절 표와
# 53-57행 체크박스, 6절 이행상태를 이 수치로 갱신한다.

saved /home/junkim/paper_ajou_dev/PanDerm/output_dir/peft_ft_comparison_aptos2019.csv
saved /home/junkim/paper_ajou_dev/PanDerm/output_dir/peft_ft_comparison_oral_diseases.csv
